In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import cv2
import os
import csv

from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from collections import Counter
from scipy.spatial import distance
from scipy.spatial import distance_matrix

import time

def showimage(image, title_=None, cmap_=None, save=None, savepath=None):
    plt.imshow(image, cmap=cmap_)
    plt.title(title_)
    plt.axis("off")
    if save:
        plt.savefig(savepath)
    plt.show()

def save_csv(savedata, savepath):
    np.savetxt(savepath, savedata, delimiter=",", fmt="%d")

def load_csv(filepath):
    return np.loadtxt(filepath, delimiter=",", dtype=np.int32)

def load_manual_markers(filepath):
    return load_csv(r"../Watershed/result\skimage_markers_.csv")

# watershedを実行して、可視化する。表示するものはmarkers, label, 統合させたもの
def show_result(markers, distance, thresh):
    # マーカーの可視化用
    mask_temp = markers.copy()
    mask_temp = mask_temp.astype(np.uint8) * 255
    # watershedの実行
    labels = watershed(-distance, markers, mask=thresh)
    # マーカーとラベルと統合して表示させる
    mask_on_labels = labels.copy()
    mask_on_labels[markers >= 1] = 255

    fig, axes = plt.subplots(ncols=3, figsize=(9, 3), sharex=True, sharey=True)
    ax = axes.ravel()
    ax[0].imshow(mask_temp)
    ax[0].set_title("markers")
    ax[1].imshow(labels)
    ax[1].set_title("labels")
    ax[1].imshow(labels)
    ax[2].imshow(mask_on_labels)
    ax[2].set_title("mask_on_labels")
    for a in ax:
        a.set_axis_off()
    fig.tight_layout()
    plt.show()

# マーカー間の距離を計算する-> 距離行列を求める
def get_marker_centroids(markers):
    centroids = []
    for label in np.unique(markers):
        if label == 0:
            # らべるが0のところはスキップ
            continue
        positions = np.argwhere(markers == label)
        centroid = positions.mean(axis=0)
        centroids.append(centroid)
    save_csv(np.array(centroids), r"..\Watershed/result/temp.csv")
    return np.array(centroids)  # shape=(n_markers, 2)

def compute_distance_matrix_scipy(centroids):
    return distance_matrix(centroids, centroids)

jpg_path = r"..\Watershed/Image/water_coins.jpg"
image = cv2.imread(jpg_path)

image_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
ret, thresh = cv2.threshold(image_gray, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

# Now we want to separate the two objects in image
# Generate the markers as local maxima of the distance to the background
distance = ndi.distance_transform_edt(thresh)
coords = peak_local_max(distance, footprint=np.ones((3, 3)), labels=thresh)
mask = np.zeros(distance.shape, dtype=bool)
mask[tuple(coords.T)] = True
markers, _ = ndi.label(mask)

# 手動のマーカーをインプットしてみる
markers_ = load_manual_markers(r"../Watershed/result\skimage_markers_.csv")

# シード値の修正
#showimage(markers)
#save_csv(markers, r"../Watershed/Result/test.csv")

#show_result(markers, distance, thresh)

# 重心取得
centroids = get_marker_centroids(markers)

# 距離行列（手動実装 or scipy使用）
dist_matrix = get_marker_centroids(centroids)
# または
# dist_matrix = compute_distance_matrix_scipy(centroids)

# 結果表示
print("距離行列:\n", dist_matrix)

"""
# 輪郭をとる
# 周囲を探索して自身と異なる値であれば-1を付与する?
Width, Height = labels_markers.shape # W:X, H:Y
for h in range(Height):
    for w in range(Width):
        if labels_markers[w, h] == -1:
            continue
        for dh in [-1, 0, 1]:
            for dw in [-1, 0, 1]:
                nh, nw = h + dh, w + dw
                if 0 <= nh < Height and 0 <= nw < Width:
                    if labels_markers[w, h] != labels_markers[nw, nh] and labels_markers[nw, nh] != -1:
                        labels_markers[w, h] = -1
"""
#labels_markers[labels_markers != -1] = 255
#labels_markers[labels_markers == -1] = 0

#showimage(labels, "skimage", None, False, "../Watershed/Result/skimage_color.png")

距離行列:
 [[17.          1.        ]
 [33.          1.        ]
 [ 0.          0.        ]
 [15.          1.        ]
 [ 1.          0.        ]
 [24.          1.        ]
 [31.          1.        ]
 [18.5         1.        ]
 [47.          1.        ]
 [ 2.          0.        ]
 [41.          1.        ]
 [14.          1.        ]
 [11.          1.        ]
 [12.          1.        ]
 [13.          1.        ]
 [46.          1.        ]
 [23.5         0.5       ]
 [ 4.          0.        ]
 [10.5         0.5       ]
 [ 6.          0.        ]
 [ 0.          1.        ]
 [45.          1.        ]
 [ 7.          0.        ]
 [ 3.          1.        ]
 [34.          1.        ]
 [ 8.          0.        ]
 [ 9.          0.        ]
 [10.          0.        ]
 [28.          1.        ]
 [11.          0.        ]
 [14.          1.        ]
 [25.          0.5       ]
 [10.66666667  0.33333333]
 [ 6.          1.        ]
 [15.          0.        ]
 [26.          1.        ]
 [16.          0.    

'\n# 輪郭をとる\n# 周囲を探索して自身と異なる値であれば-1を付与する?\nWidth, Height = labels_markers.shape # W:X, H:Y\nfor h in range(Height):\n    for w in range(Width):\n        if labels_markers[w, h] == -1:\n            continue\n        for dh in [-1, 0, 1]:\n            for dw in [-1, 0, 1]:\n                nh, nw = h + dh, w + dw\n                if 0 <= nh < Height and 0 <= nw < Width:\n                    if labels_markers[w, h] != labels_markers[nw, nh] and labels_markers[nw, nh] != -1:\n                        labels_markers[w, h] = -1\n'